# PV056 — Plant Disease Classification with Triplet Loss

**Course**: PV056 Machine Learning and Data Mining, MUNI 2026  
**Task**: Classify plant diseases (subtask a) and detect unknown diseases (subtask b) using metric learning on PlantVillage.  
**Repo**: https://github.com/MrJoeKr/pv056-project-2026

> **Runtime**: Set to **GPU** (Runtime → Change runtime type → T4 GPU) before running.  
> **Fast mode** (default): ResNet18 + reduced config, full notebook runs in ~10 min on a T4. Set `FAST_MODE = False` in section 4 to use the full ResNet50 config from the report.

---
## Contents
1. [Setup](#setup) — clone repo, install dependencies
2. [Dataset](#dataset) — download PlantVillage from Kaggle
3. [EDA](#eda) — class distribution, outlier detection
4. [Training](#training) — stratified CV with triplet loss (Fast mode toggle)
5. [Evaluation](#evaluation) — confusion matrix, Grad-CAM, UMAP, per-class F1
6. [Unknown Detection](#unknown) — Mahalanobis distance, ROC, UMAP

---
## 1. Setup <a id='setup'></a>

In [ ]:
# Clone the project repository
!git clone https://github.com/MrJoeKr/pv056-project-2026
%cd pv056-project-2026

In [ ]:
# Install PyTorch with CUDA and remaining dependencies
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q
!pip install -r requirements.txt -q

In [ ]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

---
## 2. Dataset <a id='dataset'></a>

Download the **PlantVillage** dataset (`emmarex/plantdisease`, 15 classes, ~20,638 images) straight from Kaggle.

In [ ]:
import os, zipfile

os.makedirs('data', exist_ok=True)

if not os.path.isdir('data/PlantVillage'):
    !curl -L -o ~/Downloads/plantdisease.zip https://www.kaggle.com/api/v1/datasets/download/emmarex/plantdisease
    with zipfile.ZipFile(os.path.expanduser('~/Downloads/plantdisease.zip')) as z:
        z.extractall('data')
    os.remove(os.path.expanduser('~/Downloads/plantdisease.zip'))

classes = sorted(os.listdir('data/PlantVillage'))
print(f'Dataset ready — {len(classes)} classes found')
for c in classes:
    print(' ', c)

---
## 3. Exploratory Data Analysis <a id='eda'></a>

- **R1a**: Class label distribution
- **R1b**: Pixel-level outlier detection (z-score)

In [ ]:
!python scripts/01_eda.py

In [ ]:
from IPython.display import display, Image
import glob

for path in sorted(glob.glob('results/plots/eda/*.png')):
    print(path)
    display(Image(path))

---
## 4. Outlier detection
Script: `scripts/08_outliers.py`

Three methods are run per class and compared:

**1. `knn_distance`** — Embedding-space distance  
Each image is scored by the mean distance to its 5 nearest neighbors in the embedding space (L2-normalized) computed by our model. The threshold is set automatically: elbow detection on the sorted score curve is used if a strong knee is found (strength > 0.15) at the top 10% of scores; otherwise a MAD-based threshold (median + 4.5×MAD) is applied. Catches images whose learned features are far from their class peers.

**2. `zscore`** — Pixel-level statistics  
Per-image features are the per-channel RGB means and stds (6 values total). A global z-score is computed across all images in the class; any image with at least one feature exceeding 3 standard deviations is flagged. Catches images with unusual color distributions (e.g., heavily washed-out or tinted images).

**3. `convautoencoder`** — Reconstruction error  
A single ConvAutoencoder is trained on **all** dataset images at once (not per class). At inference time, per-class reconstruction errors are sliced from the full error array. The outlier threshold is set adaptively — elbow detection on the sorted error curve is used if a strong knee is found (strength > 0.15) at the top 10% of errors; otherwise a MAD-based threshold (median + 4.5×MAD) is applied. Images the autoencoder fails to reconstruct well are flagged as outliers — these tend to be structurally different from the majority (e.g., atypical framing, heavy noise, or corrupted images).

NOTE: The following cells do not run the script directly, it shows the results instead, because training the autoencoder would take another 20 minutes.



### Global match table (outlier count agreements between methods)

| | knn_distance | zscore | convautoencoder |
|---|---|---|---|
| **knn_distance** | 1199 | 36 | 82 |
| **zscore** | 36 | 368 | 44 |
| **convautoencoder** | 82 | 44 | 1037 |

_Diagonal = total outliers flagged by that method. Off-diagonal = images flagged by both methods._

All methods intersection (total): 11

In [ ]:
from IPython.display import display, Image, HTML

IMAGES = [
    ("Outliers matched across all 3 methods", "images/outlier_consensus_gallery.png"),
]

for title, img_path in IMAGES:
    display(HTML(f"<h2>{title}</h2>"))
    display(Image(img_path))

### Conclusion

The three outlier detection methods showed poor agreement with each other. Each method flagged hundreds of images individually — 1199 by `knn_distance`, 1037 by `convautoencoder`, and 368 by `zscore` — yet pairwise intersections were small: at most 82 images agreed between `knn_distance` and `convautoencoder`, and only 36 between `knn_distance` and `zscore`. All three methods agreed on 11 images.

Manual inspection of the consensus gallery revealed that genuinely obvious outliers (corrupted files, blank images, extreme crops) were rare — roughly 3 out of ~20,000 images. The vast majority of flagged images appeared visually normal, suggesting that the methods were picking up on subtle distributional differences rather than true data quality issues.

Given the weak inter-method agreement and the near-absence of visually identifiable bad images, **we decided not to remove any images from the dataset**.

---
## 5. Training — Stratified CV <a id='training'></a>

- **R2a**: HPO was run separately (Optuna, 30 trials); best params baked into `src/config.py`
- **R2b**: Training with early stopping; plots training curves per fold

### Fast mode vs full-quality

The full run (ResNet50, 224×224, 50 epochs, 5 folds) takes ~60–90 min on a T4 — too slow for a Colab demo. The cell below writes a **`config_override.json`** that every script in this project honors, swapping in a fast configuration (ResNet18, 128×128, 10 epochs, 2 folds, mixed precision). Set `FAST_MODE = False` to run the full config used in the report.

> The authoritative numbers in the report were produced locally with ResNet50. This notebook reproduces the full pipeline end-to-end as a runnable demo; see the [GitHub repo](https://github.com/MrJoeKr/pv056-project-2026) for raw results tables and full-quality checkpoints.

In [ ]:
import json, os

FAST_MODE = True  # set to False to use Config() defaults (ResNet50, 224x224, 50 epochs, 5 folds)

override_path = 'results/tables/config_override.json'
os.makedirs(os.path.dirname(override_path), exist_ok=True)

if FAST_MODE:
    overrides = {
        'backbone': 'resnet18',
        'img_size': 128,
        'epochs': 10,
        'patience': 3,
        'n_folds': 2,
        'batch_size': 128,
        'use_amp': True,
    }
    with open(override_path, 'w') as f:
        json.dump(overrides, f, indent=2)
    print('Fast mode active — overrides written:')
    print(json.dumps(overrides, indent=2))
else:
    if os.path.exists(override_path):
        os.remove(override_path)
    print('Full mode — using Config() defaults')

In [ ]:
!python scripts/02_train.py

In [ ]:
for path in sorted(glob.glob('results/plots/training_curves_fold*.png')):
    print(path)
    display(Image(path))

cv_path = 'results/plots/cv_results.png'
if os.path.exists(cv_path):
    print(cv_path)
    display(Image(cv_path))

---
## 6. Evaluation <a id='evaluation'></a>

- **R3a**: Confusion matrix, per-class F1, Grad-CAM explainability, UMAP embedding visualization
- **R3b**: Summary table, statistical context

In [ ]:
!python scripts/04_evaluate.py

In [ ]:
import pandas as pd

# Summary table
summary = pd.read_csv('results/tables/results_summary.csv')
display(summary)

# Plots
for path in ['results/plots/evaluation/confusion_matrix.png',
             'results/plots/evaluation/per_class_f1.png',
             'results/plots/evaluation/umap_embeddings.png',
             'results/plots/evaluation/gradcam_samples.png']:
    if os.path.exists(path):
        print(path)
        display(Image(path))

---
## 7. Unknown Disease Detection <a id='unknown'></a>

Subtask b: `Tomato_Bacterial_spot` is excluded from training and treated as the unknown class.  
Detection uses **Mahalanobis distance** from test embeddings to class prototypes (not softmax).

Results from our run:
- AUROC: **0.9795**, PR-AUC: **0.9627**
- Mann-Whitney U p ≈ 0.00 (unknown distances stochastically larger, highly significant)

In [ ]:
!python scripts/05_unknown.py

In [ ]:
for path in sorted(glob.glob('results/plots/unknown/unknown_*.png')):
    print(path)
    display(Image(path))

---
## 8. GRAD-CAM analysis & motivation for preprocessing
After training our first model on the raw PlantVillage images, we ran Grad-CAM to inspect which image regions the network focused on when making predictions. A recurring pattern emerged: many images showed activation hotspots concentrated in the **top-left corner** rather than on the plant or leaf itself. This strongly suggests the model was exploiting a **background bias** — a dataset artifact where the camera position or studio setup correlates with the class label — rather than learning disease-relevant visual features.

To mitigate this, we added a preprocessing pipeline that:
- **Removes the background** (U2-Net-based segmentation via `rembg`), isolating the leaf/plant foreground
- **Corrects shadows** using a Guided Filter to normalise illumination
- **Stretches the L channel** (LAB colour space) to normalise brightness across images

The model was then retrained on the preprocessed images. The triplets below compare, for several example images: the **original image** (left), the **Grad-CAM heatmap from the model trained on original images** (centre), and the **Grad-CAM heatmap from the model trained on preprocessed images** (right). The shift in attention away from background corners and towards leaf structure confirms that preprocessing reduced the background bias.


In [ ]:
from IPython.display import display, Image
import glob

for path in sorted(glob.glob('images/gradcam/*')):
    print(path)
    display(Image(path))

---
## 9. Additional Data preprocessing

The preprocessing pipeline (`scripts/09_preprocessing.py`) applies shadow correction via Guided Filter, background removal via U2-Net (rembg), and L-channel histogram stretching to normalize the brightness of each image.

To speed things up download our preprocessed dataset.

In [ ]:
import os

PREPROCESSED_DATASET_FILE_ID="1_guCGWUXtjNrUxHTlj6ekquJlRsoZ6XM"

if not os.path.isdir('data/PlantVillage_preprocessed'):
    %pip install gdown
    !gdown "https://drive.google.com/uc?id=$PREPROCESSED_DATASET_FILE_ID" --output data/preprocessed_dataset.zip
    !unzip data/preprocessed_dataset.zip -d data/PlantVillage_preprocessed
    !rm data/preprocessed_dataset.zip

In [ ]:
from pathlib import Path
from IPython.display import display, Image

from src.visualization import plot_preprocessing_comparison_from_files


PREPROCESSING_VISUALIZE = [
    "Pepper__bell___Bacterial_spot/0efa6329-22f4-4bf0-a67a-17b0d5e4d2f2___NREC_B.Spot 9145.JPG",
    "Pepper__bell___healthy/0bb97c36-159d-4ee2-8b06-1fbf3f533af5___JR_HL 8345.JPG",
    "Potato___Early_blight/0ad3ba53-f01b-403b-a99d-5991eed85045___RS_Early.B 7600.JPG",
    "Tomato__Tomato_YellowLeaf__Curl_Virus/0a3d19ca-a126-4ea3-83e3-0abb0e9b02e3___YLCV_GCREC 2449.JPG",
    "Tomato_healthy/00bce074-967b-4d50-967a-31fdaa35e688___RS_HL 0223.JPG"
]

for img_name in PREPROCESSING_VISUALIZE:
    original_path = Path('data/PlantVillage') / img_name
    final_path = Path('data/PlantVillage_preprocessed') / img_name
    output_path = Path('results/plots/preprocessing') / f"comparison_{img_name}"
    plot_preprocessing_comparison_from_files(original_path, final_path, output_path)
    display(Image(output_path))

### Background removal summary
We retrained the ResNet50 model on the preprocessed (background-removed, shadow-corrected) images using the same hyperparameters as our best-performing model. Due to time constraints, only 1 fold of cross-validation was completed. The preprocessed model achieved F1 macro = 0.996, slightly below the original model's 0.998. While the raw score is marginally lower, the model is expected to be more robust — its attention is grounded in leaf morphology rather than background artifacts, making it less likely to overfit to dataset-specific shortcuts.

